# 5.4 填充与复制

## 1.填充 tf.pad(x, paddings)  
对于图片数据的高和宽、序列信号的长度，维度长度可能各不相同。为了方便网络的
并行计算，需要将不同长度的数据扩张为相同长度，之前我们介绍了通过复制的方式(tf.tile(x, [2, 
1])…)可以
增加数据的长度，但是**重复复制数据会破坏原有的数据结构**，并不适合于此处。  
通常的做法是，在需要补充长度的数据*开始或结束处填充足够数量的特定数值*，这些特定数值一般
代表了无效意义，例如 0，使得填充后的长度满足系统要求。那么这种操作就叫作填充
(Padding)。

填充操作可以通过 tf.pad(x, paddings)函数实现，参数 paddings 是包含了多个
[Left Padding,Right Padding]的嵌套方案 List，如[[0,0],[2,1],[1,2]]表示第一个维度不填
充，第二个维度左边(起始处)填充两个单元，右边(结束处)填充一个单元，第三个维度左边
填充一个单元，右边填充两个单元。

In [3]:
import tensorflow as tf
from tensorflow import keras

a = tf.constant([1, 2, 3, 4, 5, 6])  # 第一个句子
b = tf.constant([7, 8, 1, 6])  # 第二个句子
b = tf.pad(b, [[0, 2]])  # 代表第一个维度句子末尾填充2个0
b

<tf.Tensor: shape=(6,), dtype=int32, numpy=array([7, 8, 1, 6, 0, 0], dtype=int32)>

In [4]:
# 填充后句子张量形状一致，再将这 2 句子 Stack 在一起
tf.stack([a, b], axis=0)  # 堆叠合并，创建句子数维度

<tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[1, 2, 3, 4, 5, 6],
       [7, 8, 1, 6, 0, 0]], dtype=int32)>

在自然语言处理中，需要加载不同句子长度的数据集，有些句子长度较小，如仅 10 个单词，部份句子长度较长，如超过 100 个单词。为了能够保存在同一张量中，一般会选取能够覆盖大部分句子长度的阈值，如 80 个单词。对于小于 80 个单词的句子，在末尾填充相应数量的 0；对大于 80 个单词的句子，截断超过规定长度的部分单词。

In [5]:
# 以 IMDB 数据集的加载为例，我们来演示如何将不等长的句子变换为等长结构
total_words = 10000  # 设定词汇量大小
max_review_len = 80  # 设定最大句子长度
embedding_len = 100  # 设定词向量长度
# 加载IMDB数据集
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=total_words)

# 将句子填充或截断到相同长度，设置为末尾填充和末尾截断方式
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_review_len, truncating='post', padding='post')
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_review_len, truncating='post', padding='post')
print(x_train.shape, x_test.shape) # 打印等长的句子张量形状

(25000, 80) (25000, 80)


我们来介绍同时在多个维度进行填充的例子。考虑对图片的高宽维度进行填充。以
28 × 28大小的图片数据为例，如果网络层所接受的数据高宽为32 × 32，则必须将28 × 28
大小填充到32 × 32，可以选择在图片矩阵的上、下、左、右方向各填充 2 个单元

In [7]:
x = tf.random.normal([4, 28, 28, 1])
# 图片上下、左右各填充2个单元
tf.pad(x, [[0, 0], [2, 2], [2, 2], [0, 0]]).shape

TensorShape([4, 32, 32, 1])

## 2.复制 tf.tile() 
在维度变换一节，我们就介绍了通过 tf.tile()函数实现长度为 1 的维度复制的功能。
tf.tile 函数除了可以对长度为 1 的维度进行复制若干份，还可以对任意长度的维度进行复制
若干份，进行复制时会根据原来的数据次序重复复制。由于前面已经介绍过，此处仅作简
单回顾。
通过 tf.tile 函数可以在任意维度将数据重复复制多份，如 shape 为[4,32,32,3]的数据，
复制方案为 multiples=[2,3,3,1]，即通道数据不复制，高和宽方向分别复制 2 份，图片数再
复制 1 份

In [8]:
x = tf.random.normal([4, 32, 32, 3])
x = tf.tile(x, multiples=[2, 3, 3, 1])
x.shape

TensorShape([8, 96, 96, 3])